In [10]:
# import library
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import html
import csv
import time

In [7]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import html
from urllib.parse import urljoin
import os

def scrape_links(url, session=None, sleep_between_requests=0.5):
    if session is None:
        session = requests.Session()
    
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; Bot/0.1; +https://example.com/bot)" 
    }
    
    try:
        resp = session.get(url, headers=headers, timeout=15)
        resp.raise_for_status() # Cek jika ada error 404/500
        
        soup = BeautifulSoup(resp.text, "html.parser")

        # cari anchors khusus di blok tombol listing ("Lihat Detail")
        anchors = soup.select("div.listing-card__information__bottom__buttons a")
        
        # Fallback: mengambil semua elemen a yang memiliki href mengandung '/adform/'
        if not anchors:
            anchors = [a for a in soup.find_all("a", href=True) if "/adform/" in a["href"]]

        results = [] 
        seen = set()
        
        for a in anchors: 
            raw = a.get("href") 
            if not raw:
                continue
            decoded = html.unescape(raw)
            full = urljoin(resp.url, decoded)
            
            if full in seen: 
                continue
            seen.add(full)
            
            results.append({
                "text": a.get_text(strip=True),
                "href": decoded,
                "full_url": full,
                "rel": a.get("rel"),
                "title": a.get("title")
            })

        # Delay scraping agar tidak dianggap DDOS
        time.sleep(sleep_between_requests)
        return results

    except requests.exceptions.RequestException as e:
        print(f"Error scraping {url}: {e}")
        return []

if __name__ == "__main__":
    # Pastikan folder tujuan ada
    output_dir = "tim2/data"
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = f"{output_dir}/mitula_links_all_pages.csv"
    
    # URL Template (perhatikan bagian page={page_num})
    base_url = "https://rumah.mitula.co.id/find?page={page_num}&operationType=sell&propertyType=house&geoId=R9677345"

    keys = ["text", "href", "full_url", "title", "rel"]
    
    # Inisialisasi Session di luar loop untuk efisiensi
    session = requests.Session()
    
    print("Mulai scraping halaman 1 sampai 710...")

    # Membuka file CSV sekali di awal
    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader() # Tulis header kolom
        
        total_links_collected = 0

        # Loop dari 1 sampai 710 (711 tidak termasuk)
        for i in range(1, 711):
            target_url = base_url.format(page_num=i)
            print(f"Scraping Page {i}/710...", end=" ", flush=True)
            
            links = scrape_links(target_url, session=session)
            
            if links:
                writer.writerows(links)
                count = len(links)
                total_links_collected += count
                print(f"Berhasil ({count} links)")
            else:
                print("Kosong / Gagal")

    print("--- Selesai ---")
    print(f"Total links tersimpan: {total_links_collected}")
    print(f"File tersimpan di: {output_file}")

Mulai scraping halaman 1 sampai 710...
Scraping Page 1/710... Berhasil (30 links)
Scraping Page 2/710... Berhasil (30 links)
Scraping Page 3/710... Berhasil (30 links)
Scraping Page 4/710... Berhasil (30 links)
Scraping Page 5/710... Berhasil (30 links)
Scraping Page 6/710... Berhasil (30 links)
Scraping Page 7/710... Berhasil (30 links)
Scraping Page 8/710... Berhasil (30 links)
Scraping Page 9/710... Berhasil (30 links)
Scraping Page 10/710... Berhasil (30 links)
Scraping Page 11/710... Berhasil (30 links)
Scraping Page 12/710... Berhasil (30 links)
Scraping Page 13/710... Berhasil (30 links)
Scraping Page 14/710... Berhasil (30 links)
Scraping Page 15/710... Berhasil (30 links)
Scraping Page 16/710... Berhasil (30 links)
Scraping Page 17/710... Berhasil (30 links)
Scraping Page 18/710... Berhasil (30 links)
Scraping Page 19/710... Berhasil (30 links)
Scraping Page 20/710... Berhasil (30 links)
Scraping Page 21/710... Berhasil (30 links)
Scraping Page 22/710... Berhasil (30 links)
Sc

In [9]:
# import library
import pandas as pd
import glob

# mengambil direktori data
all_files = glob.glob('tim2/data/*.csv')

# membuat list kosong untuk menyimpan data yang ingin digabungkan
df_list = []

# membuat file csv dengan loop
for f in all_files:
    df = pd.read_csv(f)
    df_list.append(df)

# mengombinasikan hasil file .csv
combined_df = pd.concat(df_list, ignore_index=True)

# menampilkan dataframe
print("Combined DataFrame shape:", combined_df.shape)
display(combined_df.head())

# menyimpan data menjadi format .csv
combined_df.to_csv("combined_mitula_links.csv", index=False)
print("Combined CSV file saved as combined_mitula_links.csv")

Combined DataFrame shape: (4200, 5)


,text,href,full_url,title,rel
0,Lihat detail,/adform/24301-256-7192-4482611ff64e-b332-199d9...,https://rumah.mitula.co.id/adform/24301-256-71...,Lihat detail,['nofollow']
1,Lihat detail,/adform/24301-256-7bd7-2a216063cb14-ae8f-19ac1...,https://rumah.mitula.co.id/adform/24301-256-7b...,Lihat detail,['nofollow']
2,Lihat detail,/adform/24301-256-788b-f6e591a83dca-8b73-197f8...,https://rumah.mitula.co.id/adform/24301-256-78...,Lihat detail,['nofollow']
3,Lihat detail,/adform/24301-256-7559-1557fb6562f3-a2b8-197f7...,https://rumah.mitula.co.id/adform/24301-256-75...,Lihat detail,['nofollow']
4,Lihat detail,/adform/24301-256-7679-b6739e2e98ef-aee0-19ae3...,https://rumah.mitula.co.id/adform/24301-256-76...,Lihat detail,['nofollow']


Combined CSV file saved as combined_mitula_links.csv


In [ ]:
## link lamudi

In [30]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import html
from urllib.parse import urljoin
import os

def scrape_links(url, session=None, sleep_between_requests=1.0):
    if session is None:
        session = requests.Session()
    
    # Header diperbarui agar tidak diblokir Lamudi (403 Forbidden)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.google.com/"
    }
    
    try:
        resp = session.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
        
        soup = BeautifulSoup(resp.text, "html.parser")

        # --- BAGIAN MODIFIKASI SELECTOR LAMUDI ---
        
        # Cara 1: Mencari link pada Judul Listing (paling akurat)
        anchors = soup.select("a.ListingCell-KeyInfo-title-link")
        
        # Cara 2: Mencari link pada class js-listing-link (alternatif)
        if not anchors:
            anchors = soup.select("a.js-listing-link")
            
        # Cara 3 (Fallback): Cari berdasarkan pola URL jika class berubah
        if not anchors:
            # Ambil semua link yang mengandung '/jual/' atau '/sewa/' tapi BUKAN pagination
            anchors = [
                a for a in soup.find_all("a", href=True) 
                if ("/jual/" in a["href"] or "/sewa/" in a["href"]) 
                and "page=" not in a["href"]
                and "sort=" not in a["href"]
            ]
        # ------------------------------------------

        results = [] 
        seen = set()
        
        for a in anchors: 
            raw = a.get("href") 
            if not raw:
                continue
            
            # Bersihkan URL
            decoded = html.unescape(raw)
            full = urljoin(resp.url, decoded)
            
            # Filter tambahan: pastikan ini link detail properti, bukan link kategori
            # Link properti biasanya diakhiri dengan angka ID
            if not any(char.isdigit() for char in full[-15:]): 
                continue

            if full in seen: 
                continue
            seen.add(full)
            
            title_text = a.get("title") or a.get_text(strip=True)
            
            results.append({
                "text": a.get_text(strip=True),
                "href": decoded,
                "full_url": full,
                "rel": a.get("rel"),
                "title": title_text
            })

        # Delay scraping (Lamudi butuh delay lebih lama agar aman)
        time.sleep(sleep_between_requests)
        return results

    except requests.exceptions.RequestException as e:
        print(f"Error scraping {url}: {e}")
        return []

if __name__ == "__main__":
    # Pastikan folder tujuan ada
    output_dir = "lamudi"
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = f"{output_dir}/wonoayu_lamudi_links.csv"
    
    # URL Target Lamudi (Balongbendo)
    base_url = "https://www.lamudi.co.id/jual/jawa-timur/sidoarjo/wonoayu/rumah/rumah-kavling/?page={page_num}"

    keys = ["text", "href", "full_url", "title", "rel"]
    
    session = requests.Session()
    
    print("Mulai scraping Lamudi...")

    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        
        total_links_collected = 0

        # Loop halaman (Contoh: 1 sampai 5 dulu untuk tes)
        # Ubah range(1, 13) sesuai kebutuhan jumlah halaman yang tersedia di web
        for i in range(1, 4):
            target_url = base_url.format(page_num=i)
            print(f"Scraping Page {i}...", end=" ", flush=True)
            
            links = scrape_links(target_url, session=session)
            
            if links:
                writer.writerows(links)
                count = len(links)
                total_links_collected += count
                print(f"Berhasil ({count} links)")
            else:
                print("Kosong / Gagal (Mungkin halaman habis atau diblokir)")

    print("--- Selesai ---")
    print(f"Total links tersimpan: {total_links_collected}")
    print(f"File tersimpan di: {output_file}")

Mulai scraping Lamudi...
Scraping Page 1... Berhasil (18 links)
Scraping Page 2... Berhasil (14 links)
Scraping Page 3... Berhasil (6 links)
--- Selesai ---
Total links tersimpan: 38
File tersimpan di: lamudi/wonoayu_lamudi_links.csv


In [56]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import html
from urllib.parse import urljoin
import os

def scrape_links(url, session=None, sleep_between_requests=0.5):
    if session is None:
        session = requests.Session()
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36" 
    }
    
    try:
        resp = session.get(url, headers=headers, timeout=15)
        resp.raise_for_status() # Cek jika ada error 404/500
        
        soup = BeautifulSoup(resp.text, "html.parser")

        # --- BAGIAN YANG DIUBAH (Selector disesuaikan dengan Lamudi) ---
        # Mengambil anchor <a> yang berada langsung di dalam div class 'snippet__content'
        anchors = soup.select("div.snippet__content > a")
        
        # Fallback: Jika tidak ditemukan, coba cari anchor pembungkus utama card
        if not anchors:
            anchors = soup.select("div[data-test='normal-listing'] > a")
        # ---------------------------------------------------------------

        results = [] 
        seen = set()
        
        for a in anchors: 
            raw = a.get("href") 
            if not raw:
                continue
            decoded = html.unescape(raw)
            full = urljoin(resp.url, decoded)
            
            if full in seen: 
                continue
            seen.add(full)
            
            results.append({
                "text": a.get_text(strip=True),
                "href": decoded,
                "full_url": full,
                "rel": a.get("rel"),
                "title": a.get("title")
            })

        # Delay scraping agar tidak dianggap DDOS
        time.sleep(sleep_between_requests)
        return results

    except requests.exceptions.RequestException as e:
        print(f"Error scraping {url}: {e}")
        return []

if __name__ == "__main__":
    # Pastikan folder tujuan ada
    output_dir = "lamudi"
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = f"{output_dir}/wonoayu_links_all_page3.csv"
    
    # URL Template (perhatikan bagian page={page_num})
    base_url = "https://www.lamudi.co.id/jual/jawa-timur/sidoarjo/wonoayu/rumah/rumah-kavling/?page=3"

    keys = ["text", "href", "full_url", "title", "rel"]
    
    # Inisialisasi Session di luar loop untuk efisiensi
    session = requests.Session()
    
    print("Mulai scraping halaman 1 sampai 710...")

    # Membuka file CSV sekali di awal
    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader() # Tulis header kolom
        
        total_links_collected = 0

        # Loop dari 1 sampai 710 (711 tidak termasuk)
        for i in range(1, 2):
            target_url = base_url.format(page_num=i)
            print(f"Scraping Page {i}/710...", end=" ", flush=True)
            
            links = scrape_links(target_url, session=session)
            
            if links:
                writer.writerows(links)
                count = len(links)
                total_links_collected += count
                print(f"Berhasil ({count} links)")
            else:
                print("Kosong / Gagal")

    print("--- Selesai ---")
    print(f"Total links tersimpan: {total_links_collected}")
    print(f"File tersimpan di: {output_file}")

Mulai scraping halaman 1 sampai 710...
Scraping Page 1/710... Berhasil (30 links)
--- Selesai ---
Total links tersimpan: 30
File tersimpan di: lamudi/wonoayu_links_all_page3.csv


In [57]:
import pandas as pd
import glob
import os

# 1. Konfigurasi Path
# Sesuaikan dengan lokasi folder di gambar Anda (/tim2/lamudi/)
folder_path = 'lamudi' 
output_filename = 'gabungan_semua_kecamatan_lamudi.csv'

# 2. Ambil semua file berakhiran .csv di folder tersebut
# os.path.join memastikan path terbaca dengan benar di OS apapun
search_pattern = os.path.join(folder_path, "*.csv") 
all_files = glob.glob(search_pattern)

print(f"Ditemukan {len(all_files)} file CSV di folder '{folder_path}'\n")

# List untuk menampung dataframe sementara
df_list = []

# 3. Loop untuk membaca setiap file
for filename in all_files:
    try:
        # Baca CSV
        df = pd.read_csv(filename)
        
        # --- OPSIONAL: Ekstrak nama kecamatan dari nama file ---
        # Contoh nama file: "tim2/lamudi/porong_lamudi_links.csv"
        # Kita ambil nama file aslinya -> "porong_lamudi_links.csv"
        base_name = os.path.basename(filename)
        
        # Kita ambil kata pertama sebelum tanda underscore '_' (misal: "porong")
        nama_kecamatan = base_name.split('_')[0].title()
        
        # Buat kolom baru agar identitas data tidak hilang
        df['asal_kecamatan'] = nama_kecamatan
        # -------------------------------------------------------

        df_list.append(df)
        print(f"✔ Berhasil membaca: {base_name} ({len(df)} baris) -> Kec. {nama_kecamatan}")
        
    except Exception as e:
        print(f"❌ Gagal membaca {filename}: {e}")

# 4. Proses Penggabungan (Concatenation)
if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Simpan hasil gabungan
    combined_df.to_csv(output_filename, index=False)
    
    print("-" * 40)
    print("PROSES SELESAI")
    print("-" * 40)
    print(f"Total file digabung : {len(df_list)}")
    print(f"Total baris data    : {len(combined_df)}")
    print(f"Lokasi file output  : {output_filename}")
    
    # Tampilkan 5 data teratas untuk pengecekan
    print("\nContoh 5 data teratas:")
    display(combined_df.head())
    
    # Cek sebaran data per kecamatan
    print("\nJumlah data per kecamatan:")
    print(combined_df['asal_kecamatan'].value_counts())
else:
    print("Tidak ada file CSV yang ditemukan atau list kosong.")

Ditemukan 12 file CSV di folder 'lamudi'

✔ Berhasil membaca: krembung_links_all_pages2.csv (30 baris) -> Kec. Krembung
✔ Berhasil membaca: wonoayu_links_all_page3.csv (30 baris) -> Kec. Wonoayu
✔ Berhasil membaca: prambon_links_all_pages.csv (30 baris) -> Kec. Prambon
✔ Berhasil membaca: wonoayu_links_all_pages1.csv (30 baris) -> Kec. Wonoayu
✔ Berhasil membaca: tanggulangin_links_all_pages.csv (22 baris) -> Kec. Tanggulangin
✔ Berhasil membaca: balongbendo_links_all_pages.csv (390 baris) -> Kec. Balongbendo
✔ Berhasil membaca: krembung_links_all_pages3.csv (30 baris) -> Kec. Krembung
✔ Berhasil membaca: jabon_links_all_pages.csv (30 baris) -> Kec. Jabon
✔ Berhasil membaca: porong_links_all_pages.csv (13 baris) -> Kec. Porong
✔ Berhasil membaca: krembung_links_all_pages1.csv (30 baris) -> Kec. Krembung
✔ Berhasil membaca: tarik_links_all_pages.csv (24 baris) -> Kec. Tarik
✔ Berhasil membaca: wonoayu_links_all_page2.csv (30 baris) -> Kec. Wonoayu
---------------------------------------

,text,href,full_url,title,rel,asal_kecamatan
0,"Rumah Dijual di KrembungKrembung, SidoarjoShoj...",/jual/jawa-timur/sidoarjo/new-project-shojilan...,https://www.lamudi.co.id/jual/jawa-timur/sidoa...,NaN,NaN,Krembung
1,"Rumah Dijual di KrembungKrembung, SidoarjoDiju...",/properti/41032-73-97a0cd295dfb-31ef-19aaed9-8...,https://www.lamudi.co.id/properti/41032-73-97a...,NaN,NaN,Krembung
2,"Rumah Dijual di KrembungKrembung, SidoarjoDIJU...",/jual/jawa-timur/sidoarjo/rumah-mentari-bumi-s...,https://www.lamudi.co.id/jual/jawa-timur/sidoa...,NaN,NaN,Krembung
3,"Rumah Dijual di KrembungKrembung, SidoarjoBARU...",/jual/jawa-timur/sidoarjo/rumah-bumi-cabean-as...,https://www.lamudi.co.id/jual/jawa-timur/sidoa...,NaN,NaN,Krembung
4,"Rumah Dijual di KrembungKrembung, SidoarjoRUMA...",/jual/jawa-timur/sidoarjo/griya-hati-hijau-und...,https://www.lamudi.co.id/jual/jawa-timur/sidoa...,NaN,NaN,Krembung



Jumlah data per kecamatan:
asal_kecamatan
Balongbendo     390
Krembung         90
Wonoayu          90
Prambon          30
Jabon            30
Tarik            24
Tanggulangin     22
Porong           13
Name: count, dtype: int64
